In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content/drive/MyDrive/Group1-23127084-23127250-23127493/1-source-final

# Emotion Classification - POSTER (Original) vs Improvement

Upload a face image, then click **Predict** to compare:
- **POSTER (Original)**: `checkpoint-save/rafdb_best.pth`
- **Improvement**: `checkpoint/rafdb_improvement_best.pth`

**Visualization**: For each model, you'll see attention maps from both streams:
- **Image Stream** (IR50 backbone): Captures facial features
- **Landmark Stream** (MobileFaceNet): Focuses on facial landmarks (shape, structure)

In [ ]:
import io
import os
import sys
import warnings
from pathlib import Path
from typing import Dict, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
import ipywidgets as widgets
from IPython.display import display, clear_output
from PIL import Image, ImageOps
from torchvision import transforms

warnings.filterwarnings("ignore")
plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["axes.facecolor"] = "white"

In [ ]:
def find_project_root(start: Optional[Path] = None) -> Path:
    current = start or Path.cwd()
    if current.is_file():
        current = current.parent
    candidates = [current, *current.parents]
    for candidate in candidates:
        if (candidate / "models").exists() and (candidate / "checkpoint").exists():
            return candidate
    return current


PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from models.emotion_hyp import pyramid_trans_expr as PosterEmotionModel
from models.improvement_emotion_hyp import pyramid_trans_expr as ImprovementEmotionModel
from utils import load_pretrained_weights

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Project root: {PROJECT_ROOT}")
print(f"Device: {DEVICE}")

EMOTION_NAMES = {
    0: "Surprise",
    1: "Fear",
    2: "Disgust",
    3: "Happy",
    4: "Sad",
    5: "Anger",
    6: "Neutral",
}

CHECKPOINTS = {
    "POSTER (Original)": {
        "builder": PosterEmotionModel,
        "path": PROJECT_ROOT / "checkpoint-save" / "rafdb_best.pth",
    },
    "Improvement": {
        "builder": ImprovementEmotionModel,
        "path": PROJECT_ROOT / "checkpoint" / "rafdb_improvement_best.pth",
    },
}

MODEL_CACHE: Dict[str, torch.nn.Module] = {}
CAM_CACHE: Dict[str, "DualStreamCAM"] = {}

PREPROCESS = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)

class DualStreamCAM:
    """CAM for both image stream (ir_back.body) and landmark stream (face_landback)."""
    def __init__(self, model: torch.nn.Module):
        self.model = model
        self.img_activations = None
        self.img_gradients = None
        self.lm_activations = None
        self.lm_gradients = None
        
        self._img_handle = model.ir_back.body.register_forward_hook(self._img_forward_hook)
        self._lm_handle = model.face_landback.register_forward_hook(self._lm_forward_hook)

    def _img_forward_hook(self, module, input, output):
        # output is a tensor from ir_back.body
        self.img_activations = output
        if isinstance(output, torch.Tensor) and output.requires_grad:
            output.register_hook(self._save_img_gradients)

    def _lm_forward_hook(self, module, input, output):
        # output from face_landback is a tuple: (landmarks, features)
        # We want the features (second element)
        if isinstance(output, tuple):
            output = output[1] if len(output) > 1 else output[0]
        self.lm_activations = output
        if isinstance(output, torch.Tensor) and output.requires_grad:
            output.register_hook(self._save_lm_gradients)

    def _save_img_gradients(self, gradients):
        self.img_gradients = gradients

    def _save_lm_gradients(self, gradients):
        self.lm_gradients = gradients

    def _compute_cam(self, activations, gradients):
        """Compute CAM from activations and gradients."""
        if activations is None or gradients is None:
            return np.zeros((224, 224), dtype=np.float32)
        
        activations = activations.detach()
        gradients = gradients.detach()
        
        # Handle token features (B, N, C) and conv features (B, C, H, W)
        if activations.dim() == 3:  # (B, N, C)
            cam = torch.sum(gradients * activations, dim=2)  # (B, N)
            cam = torch.relu(cam)
            n_tokens = cam.shape[1]
            side = int(np.sqrt(n_tokens))
            if side * side == n_tokens:
                cam = cam.view(cam.shape[0], 1, side, side)
            else:
                cam = cam.mean(dim=1, keepdim=True).view(cam.shape[0], 1, 1, 1)
            cam = F.interpolate(cam, size=(224, 224), mode="bilinear", align_corners=False)
            cam = cam[0, 0].cpu().numpy()
        else:  # (B, C, H, W)
            weights = gradients.mean(dim=(2, 3), keepdim=True)
            cam = torch.sum(weights * activations, dim=1, keepdim=True)
            cam = torch.relu(cam)
            cam = F.interpolate(cam, size=(224, 224), mode="bilinear", align_corners=False)
            cam = cam[0, 0].cpu().numpy()
        
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        
        # Ensure 2D and resize to (224, 224)
        if cam.ndim == 0:
            cam = np.ones((224, 224), dtype=np.float32) * cam.item()
        elif cam.ndim == 1:
            cam = np.tile(cam, (224, 224 // len(cam)))
        
        cam_img = Image.fromarray((cam * 255).astype(np.uint8))
        cam_img = cam_img.resize((224, 224), Image.BILINEAR)
        cam = np.asarray(cam_img, dtype=np.float32) / 255.0
        
        return cam

    def __call__(self, input_tensor: torch.Tensor, class_idx: Optional[int] = None):
        self.model.zero_grad(set_to_none=True)
        self.img_activations = None
        self.img_gradients = None
        self.lm_activations = None
        self.lm_gradients = None

        # Landmark branch is frozen; enable gradient flow via input for CAM.
        cam_input = input_tensor.clone().detach().requires_grad_(True)

        with torch.set_grad_enabled(True):
            logits, _ = self.model(cam_input)
            if class_idx is None:
                class_idx = int(torch.argmax(logits, dim=1).item())

            if logits.requires_grad:
                score = logits[:, class_idx].sum()
                score.backward()

        cam_img = self._compute_cam(self.img_activations, self.img_gradients)
        cam_lm = self._compute_cam(self.lm_activations, self.lm_gradients)

        return cam_img, cam_lm, logits.detach()

    def close(self):
        self._img_handle.remove()
        self._lm_handle.remove()

class ModelResult:
    def __init__(self, label: str, confidence: float, heatmap_img: np.ndarray, heatmap_lm: np.ndarray, logits: torch.Tensor):
        self.label = label
        self.confidence = confidence
        self.heatmap_img = heatmap_img
        self.heatmap_lm = heatmap_lm
        self.logits = logits


def load_model(model_key: str) -> torch.nn.Module:
    if model_key in MODEL_CACHE:
        return MODEL_CACHE[model_key]

    spec = CHECKPOINTS[model_key]
    checkpoint_path = spec["path"]
    if not checkpoint_path.exists():
        raise FileNotFoundError(f"Missing checkpoint: {checkpoint_path}")

    model = spec["builder"](img_size=224, num_classes=7, type="large")
    checkpoint = torch.load(str(checkpoint_path), map_location="cpu", weights_only=False)
    state_dict = checkpoint["model_state_dict"] if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint else checkpoint
    model = load_pretrained_weights(model, state_dict)
    model = model.to(DEVICE)
    model.eval()
    MODEL_CACHE[model_key] = model
    return model


def get_cam(model_key: str) -> DualStreamCAM:
    if model_key in CAM_CACHE:
        return CAM_CACHE[model_key]

    model = load_model(model_key)
    cam = DualStreamCAM(model)
    CAM_CACHE[model_key] = cam
    return cam


def extract_uploaded_image(file_upload: widgets.FileUpload) -> Tuple[Optional[bytes], Optional[str]]:
    value = file_upload.value
    if not value:
        return None, None

    first_item = None
    if isinstance(value, dict):
        first_item = next(iter(value.values())) if value else None
    elif isinstance(value, (list, tuple)):
        first_item = value[0] if value else None
    else:
        first_item = value

    if first_item is None:
        return None, None

    if isinstance(first_item, dict):
        content = first_item.get("content") or first_item.get("data")
        name = first_item.get("name") or first_item.get("metadata", {}).get("name") or "uploaded_image"
        return content, name

    content = getattr(first_item, "content", None) or getattr(first_item, "data", None)
    name = getattr(first_item, "name", None) or "uploaded_image"
    return content, name


def load_image_from_upload(file_upload: widgets.FileUpload) -> Tuple[Image.Image, str]:
    content, name = extract_uploaded_image(file_upload)
    if content is None:
        raise ValueError("Please upload a JPEG or PNG image first.")

    image = Image.open(io.BytesIO(content)).convert("RGB")
    image = ImageOps.exif_transpose(image)
    return image, name


def overlay_heatmap(image: Image.Image, heatmap: np.ndarray, alpha: float = 0.45) -> np.ndarray:
    base = np.asarray(image.resize((224, 224))).astype(np.float32) / 255.0
    heatmap = np.clip(heatmap, 0.0, 1.0)
    colored = plt.get_cmap("jet")(heatmap)[..., :3]
    overlay = np.clip((1.0 - alpha) * base + alpha * colored, 0.0, 1.0)
    return overlay


def predict_one(model_key: str, image: Image.Image) -> ModelResult:
    model = load_model(model_key)
    cam = get_cam(model_key)
    tensor = PREPROCESS(image).unsqueeze(0).to(DEVICE)

    with torch.enable_grad():
        cam_img, cam_lm, logits = cam(tensor, class_idx=None)

    probabilities = torch.softmax(logits, dim=1)[0].detach().cpu().numpy()
    class_idx = int(np.argmax(probabilities))
    label = EMOTION_NAMES.get(class_idx, f"Class {class_idx}")
    confidence = float(probabilities[class_idx])
    return ModelResult(label=label, confidence=confidence, heatmap_img=cam_img, heatmap_lm=cam_lm, logits=logits)


uploader = widgets.FileUpload(
    accept="image/*",
    multiple=False,
    description="Upload face image",
    icon="upload",
)

predict_button = widgets.Button(
    description="Predict",
    button_style="primary",
    icon="search",
    layout=widgets.Layout(width="150px"),
)

status = widgets.HTML(
    "<b>Status:</b> Upload a JPEG/PNG face image, then click Predict."
)
output = widgets.Output()
summary = widgets.HTML()


def render_results(original_image: Image.Image, poster_result: ModelResult, improvement_result: ModelResult, image_name: str):
    """Render 2x5 grid: Original | POSTER img+lm | Improvement img+lm."""
    original_rgb = np.asarray(original_image.resize((224, 224)))
    poster_img_overlay = overlay_heatmap(original_image, poster_result.heatmap_img)
    poster_lm_overlay = overlay_heatmap(original_image, poster_result.heatmap_lm)
    improvement_img_overlay = overlay_heatmap(original_image, improvement_result.heatmap_img)
    improvement_lm_overlay = overlay_heatmap(original_image, improvement_result.heatmap_lm)

    fig, axes = plt.subplots(2, 5, figsize=(22, 9))

    # Row 1: Original + POSTER(img/lm) + Improvement(img/lm)
    axes[0, 0].imshow(original_rgb)
    axes[0, 0].set_title("Original Image", fontsize=12, fontweight="bold")
    axes[0, 0].axis("off")

    axes[0, 1].imshow(poster_img_overlay)
    axes[0, 1].set_title(f"POSTER (Original)\nImage Stream\n{poster_result.label}", fontsize=11)
    axes[0, 1].axis("off")

    axes[0, 2].imshow(poster_lm_overlay)
    axes[0, 2].set_title("POSTER (Original)\nLandmark Stream", fontsize=11)
    axes[0, 2].axis("off")

    axes[0, 3].imshow(improvement_img_overlay)
    axes[0, 3].set_title(f"Improvement\nImage Stream\n{improvement_result.label}", fontsize=11)
    axes[0, 3].axis("off")

    axes[0, 4].imshow(improvement_lm_overlay)
    axes[0, 4].set_title("Improvement\nLandmark Stream", fontsize=11)
    axes[0, 4].axis("off")

    # Row 2: Confidence scores
    for i in range(5):
        axes[1, i].axis("off")

    # Add confidence text in row 2
    axes[1, 1].text(0.5, 0.5, f"Confidence:\n{poster_result.confidence * 100:.1f}%", 
                    ha="center", va="center", fontsize=12, fontweight="bold",
                    transform=axes[1, 1].transAxes)
    axes[1, 3].text(0.5, 0.5, f"Confidence:\n{improvement_result.confidence * 100:.1f}%",
                    ha="center", va="center", fontsize=12, fontweight="bold",
                    transform=axes[1, 3].transAxes)

    fig.suptitle(f"RAF-DB Dual-Stream Emotion Demo: {image_name}", fontsize=14, fontweight="bold")
    fig.tight_layout()
    return fig


def on_predict_clicked(_):
    with output:
        clear_output(wait=True)
        try:
            image, image_name = load_image_from_upload(uploader)
            status.value = f"<b>Status:</b> Running prediction for <code>{image_name}</code>..."

            poster_result = predict_one("POSTER (Original)", image)
            improvement_result = predict_one("Improvement", image)

            fig = render_results(image, poster_result, improvement_result, image_name)
            display(fig)
            plt.close(fig)

            summary.value = f"""
            <div style='display:grid;grid-template-columns:1fr 1fr;gap:12px;margin-top:12px;'>
              <div style='border:1px solid #ddd;border-radius:12px;padding:12px;background:#fafafa;'>
                <div style='font-size:14px;font-weight:700;'>POSTER (Original)</div>
                <div style='font-size:18px;margin-top:6px;'>{poster_result.label}</div>
                <div style='color:#444;margin-top:4px;'>Confidence: {poster_result.confidence * 100:.1f}%</div>
              </div>
              <div style='border:1px solid #ddd;border-radius:12px;padding:12px;background:#fafafa;'>
                <div style='font-size:14px;font-weight:700;'>Improvement</div>
                <div style='font-size:18px;margin-top:6px;'>{improvement_result.label}</div>
                <div style='color:#444;margin-top:4px;'>Confidence: {improvement_result.confidence * 100:.1f}%</div>
              </div>
            </div>
            """
            display(summary)
            status.value = "<b>Status:</b> Prediction finished."
        except Exception as exc:
            status.value = f"<b>Status:</b> Error: {exc}"
            raise


predict_button.on_click(on_predict_clicked)

ui = widgets.VBox(
    [
        widgets.HTML(
            """
            <div style='padding:14px 16px;border-radius:14px;background:linear-gradient(135deg,#f6f8ff,#eef7f3);border:1px solid #dde6ff;'>
              <div style='font-size:24px;font-weight:800;margin-bottom:6px;'>Emotion Classification Demo</div>
              <div style='font-size:14px;color:#333;'>Upload one face image to compare POSTER (Original) vs Improvement with dual-stream attention.</div>
            </div>
            """
        ),
        widgets.HBox([uploader, predict_button]),
        status,
        output,
    ],
    layout=widgets.Layout(gap="10px"),
)

display(ui)